### **Spark streaming Vs Autoloader**
- So traditional file streaming can suffer from **manual setup**—it needs you to explicitly define sources, checkpoints, and sinks. 
- Second, it often relies on a polling mechanism (means the system repeatedly checks a location—like a directory—at set intervals to see if new files have arrived), which can introduce** latency** and potentially miss bursts of data. 
- Third, it requires you to **handle schema evolution manually**, so any changes in data structure need extra logic. 
- Fourth, the driver keeps a growing map of processed files, which can lead to **memory bottlenecks when scaling**. 
- And finally, it doesn't automatically handle bursty or unpredictable file arrivals, so you end up with potential backpressure or lag.

### Autoloader
- Auto Loader offers seamless **incremental processing**—new files are detected as soon as they arrive. 
- Second, it **handles schema evolution automatically**, so if your data structure changes, it adapts. 
- Third, it **scales naturally** with cloud storage—no single bottleneck driver. Auto Loader uses a **distributed storage layer**, often built on something like Delta or even a key-value store—like RocksDB—to keep track of the state. This allows it to persist file metadata, offsets, and progress in a distributed way, so it doesn't overwhelm the driver.
- Fourth, it’s **event-driven**, so you don’t have to rely on periodic polling. 
- Fifth, it requires **less manual setup**—no custom logic for tracking files. 
- And finally, it **integrates natively with Delta Lake**, which gives you ACID transactions, reliability, and a solid data lakehouse approach. So, overall, it’s all about easier scaling, less manual overhead, and built-in resiliency.

In [0]:
df_customers = spark.readStream.format("CloudFiles").option(
    "cloudFiles.format","json").option(
    "cloudFiles.schemaLocation","/Volumes/gizmobox/bronze/customers_autoloader/_schema").option(
    "cloudFiles.inferColumnTypes","true").option(
    "cloudFiles.schemaHints","created_timestamp TIMESTAMP, date_of_birth DATE, member_since DATE").option(
    "cloudFiles.schemaEvolutionMode","addNewColumns").option(
    "cloudFiles.useNotifications","true").option(
        "pathGlobFilter","customers_2024_*.json"  # select specific files based on pattern specified.
    ).load(
    "/Volumes/gizmobox/landing/streaming_data/")
    
#.option("cloudFiles.schemaEvolutionMode","rescue") # modifications will go it a rescue column and not be reflected in the schema

In [0]:
customers_query = df_customers.writeStream.format("delta").option(
    "checkpointLocation",
    "/Volumes/gizmobox/bronze/checkpoints/customers_autoloader"
).option(
    "mergeSchema",
    "true")   # to add new columns
.trigger(
    availableNow=True
).toTable("gizmobox.bronze.customers_autoloader")

In [0]:
%sql
select * from gizmobox.bronze.customers_autoloader

![image_1773176856672.png](./image_1773176856672.png "image_1773176856672.png")

![image_1773176970663.png](./image_1773176970663.png "image_1773176970663.png")

Spark Structured Streaming, the combination of a write-ahead log (or WAL) and checkpointing is what ensures fault tolerance. Here’s how it works:

First, the write-ahead log stores all the incoming records before they’re processed. So, every event is written to this log as soon as it arrives, before any transformations or actions happen. Then, checkpointing keeps track of the progress of the stream—meaning it records exactly which data has been processed so far, what offsets or batch IDs were handled, and the exact state of the processing.

If a failure occurs—like a machine crash—Spark can recover by reading the write-ahead log to re-ingest any lost events, and it uses the checkpoint to restore the progress—so Spark knows exactly which data was successfully processed and which needs reprocessing. This way, you avoid duplicates, and you never lose any records. It’s really a combination of these two mechanisms—WAL ensures all input is safely logged, and checkpointing ensures that progress is remembered—so the whole pipeline can resume smoothly from where it left off.